# BirdCLEF+ 2026: Testing Data Notebook

This notebook inspects the **test data** and checks that a submission built from it is valid. It mirrors how the EoS.9 ensemble reads its test files. It covers:

1. Locating the test soundscapes, with the same **dry-run fallback** EoS.9 uses when the hidden test set is not mounted
2. File-name metadata (site, date, hour)
3. Audio header checks (sample rate, channels, duration)
4. The `row_id` values a valid submission must contain
5. How the test files compare with the training soundscapes (sites and hours the models have never seen)
6. Optional: whether the Perch ONNX model is available
7. A **submission validator** and a **row-id comparison** for the per-model files that EoS.9 blends
8. Why some outputs depend on the composition of the test set

**Requirements.** `pandas`, `numpy`, `matplotlib` and `soundfile`. On Kaggle, attach the BirdCLEF+ 2026 competition data. Elsewhere, set the `BIRDCLEF_DIR` environment variable to the folder that contains `sample_submission.csv` and `taxonomy.csv`.

**Important.** On Kaggle the real test files are only mounted while a submission is being scored. In an interactive session `test_soundscapes/` is normally empty, and the local `sample_submission.csv` is a short placeholder (3 rows in the EoS.9 logs). This notebook therefore works in two modes and tells you which one it is using.

In [ ]:
import os, re, json, ast, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

# ---- constants (same values the EoS.9 notebook uses) ----
SR         = 32_000                 # sample rate in Hz
WINDOW_SEC = 5                      # one prediction window
FILE_SEC   = 60                     # one soundscape file
N_WINDOWS  = FILE_SEC // WINDOW_SEC # 12 windows per file
SEED       = 42

# File names look like BC2026_Train_0001_S08_20250606_030007.ogg
FNAME_RE = re.compile(r"BC2026_(Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")

def parse_fname(name):
    m = FNAME_RE.match(name)
    if not m:
        return {"split": None, "file_id": None, "site": "unknown", "date": pd.NaT,
                "time_utc": None, "hour_utc": -1, "month": -1}
    split, file_id, site, ymd, hms = m.groups()
    dt = pd.to_datetime(ymd, format="%Y%m%d", errors="coerce")
    return {"split": split, "file_id": file_id, "site": site, "date": dt, "time_utc": hms,
            "hour_utc": int(hms[:2]), "month": int(dt.month) if pd.notna(dt) else -1}

def find_competition_dir():
    # Set BIRDCLEF_DIR to point at the data when running outside Kaggle.
    candidates = []
    if os.environ.get("BIRDCLEF_DIR"):
        candidates.append(Path(os.environ["BIRDCLEF_DIR"]))
    candidates += [Path("/kaggle/input/competitions/birdclef-2026"), Path("/kaggle/input/birdclef-2026")]
    for p in candidates:
        if (p / "sample_submission.csv").exists() and (p / "taxonomy.csv").exists():
            return p
    root = Path("/kaggle/input")
    if root.exists():
        for p in root.rglob("sample_submission.csv"):
            if (p.parent / "taxonomy.csv").exists():
                return p.parent
    raise FileNotFoundError("BirdCLEF competition data not found. Attach the competition data "
                            "or set the BIRDCLEF_DIR environment variable.")

BASE = find_competition_dir()
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Competition data:", BASE)
print("Contents        :", sorted(p.name for p in BASE.iterdir())[:20])
print("Outputs go to   :", OUT_DIR.resolve())

In [ ]:
DRYRUN_N = 20   # number of training soundscapes used when the hidden test set is not mounted (EoS.9 uses 20)

## 1. Locate the test data

If `test_soundscapes/*.ogg` exists, those files are used. If it does not, the notebook falls back to the first `DRYRUN_N` files of `train_soundscapes/`, exactly as the EoS.9 model cells do. Everything below then describes training files, not real test files, and the results should be read with that in mind.

In [ ]:
test_dir = BASE / "test_soundscapes"
test_paths = sorted(test_dir.glob("*.ogg")) if test_dir.exists() else []
DRY_RUN = len(test_paths) == 0

if DRY_RUN:
    fallback_dir = BASE / "train_soundscapes"
    test_paths = sorted(fallback_dir.glob("*.ogg"))[:DRYRUN_N] if fallback_dir.exists() else []
    print(f"MODE: DRY RUN. Hidden test set not mounted; using the first {len(test_paths)} train soundscapes.")
else:
    print(f"MODE: REAL TEST DATA. Found {len(test_paths)} test files in {test_dir}")

assert len(test_paths) > 0, "No audio files found in test_soundscapes/ or train_soundscapes/."
print("First files:", [p.name for p in test_paths[:3]])

## 2. File-name metadata

Every file name follows `BC2026_<Train|Test>_<file id>_<site>_<date>_<UTC time>.ogg`. The site and the UTC hour are used by the models (priors and site/hour embeddings), so their coverage matters.

In [ ]:
files = pd.DataFrame([{"filename": p.name, "stem": p.stem, **parse_fname(p.name)} for p in test_paths])
bad = files[files["site"] == "unknown"]
print(f"Files: {len(files)}  |  names that do not match the pattern: {len(bad)}")
if len(bad):
    display(bad[["filename"]])

display(files.head(10))
print("Sites:", files["site"].value_counts().to_dict())
print("Date range:", files["date"].min(), "to", files["date"].max())

fig, axes = plt.subplots(1, 2, figsize=(9, 2.8))
files["site"].value_counts().plot.bar(ax=axes[0], color="#1F3A5F"); axes[0].set_title("Files per site"); axes[0].set_ylabel("Files")
files["hour_utc"].value_counts().reindex(range(24), fill_value=0).plot.bar(ax=axes[1], color="#2A7F8E")
axes[1].set_title("Files per UTC hour"); axes[1].set_xlabel("hour_utc")
plt.tight_layout(); plt.show()

## 3. Audio header checks

EoS.9 reads each file as mono 32 kHz audio, **pads or trims it to exactly 60 seconds**, and splits it into 12 windows of 5 seconds. One model cell raises an error if the sample rate is not 32 kHz. This section reads only the headers, so it is fast, and flags anything that would be padded, trimmed, down-mixed or rejected.

In [ ]:
import soundfile as sf

rows = []
for p in test_paths:
    try:
        i = sf.info(str(p))
        rows.append({"filename": p.name, "sample_rate": i.samplerate, "channels": i.channels,
                     "duration_s": i.frames / i.samplerate, "format": i.format, "error": ""})
    except Exception as e:
        rows.append({"filename": p.name, "sample_rate": np.nan, "channels": np.nan, "duration_s": np.nan, "format": "", "error": str(e)[:80]})
audio = pd.DataFrame(rows)

audio["flag_bad_sample_rate"] = audio["sample_rate"] != SR
audio["flag_not_60s"]         = ~np.isclose(audio["duration_s"], FILE_SEC, atol=0.05)
audio["flag_not_mono"]        = audio["channels"] != 1
audio["flag_unreadable"]      = audio["error"] != ""
flag_cols = [c for c in audio.columns if c.startswith("flag_")]

print(f"Files checked: {len(audio)}")
display(audio[flag_cols].sum().rename("files flagged").to_frame())
if audio[flag_cols].any(axis=1).any():
    print("Flagged files:")
    display(audio[audio[flag_cols].any(axis=1)])
print("Note: in DRY RUN mode or in this small test the durations may be shorter than 60 s." if DRY_RUN else "")

## 4. The `row_id` values a valid submission needs

Each file produces 12 rows, one per 5-second window. The `row_id` is the file name without `.ogg`, an underscore, and the **end** second of the window (5, 10, ..., 60). EoS.9 asserts that the set of `row_id` values equals the set in `sample_submission.csv`.

In [ ]:
expected_ids = [f"{p.stem}_{WINDOW_SEC * (k + 1)}" for p in test_paths for k in range(N_WINDOWS)]
sample = pd.read_csv(BASE / "sample_submission.csv")
sample_ids = set(sample["row_id"].astype(str))

print(f"Files: {len(test_paths)}  ->  expected rows: {len(expected_ids)}  (= files x {N_WINDOWS})")
print(f"Rows in sample_submission.csv: {len(sample):,}")
print("Example row_ids:", expected_ids[:3], "...", expected_ids[N_WINDOWS - 1])

missing_from_sample = sorted(set(expected_ids) - sample_ids)
extra_in_sample     = sorted(sample_ids - set(expected_ids))
print(f"Expected but not in sample_submission: {len(missing_from_sample):,}")
print(f"In sample_submission but not expected: {len(extra_in_sample):,}")
if not missing_from_sample and not extra_in_sample:
    print("\nrow_id sets are identical.")
elif DRY_RUN:
    print("\nA mismatch is expected in DRY RUN mode: the expected ids come from training files, while the local "
          "sample_submission.csv is a short placeholder.")
elif len(sample) < len(expected_ids):
    print("\nThe sample_submission.csv here has fewer rows than the test files imply, so it looks like a short placeholder. "
          "During real scoring the two sets should be identical.")
else:
    print("\nThe two sets differ. Check the file names in test_soundscapes/ against sample_submission.csv.")

## 5. Do the test files look like the training soundscapes?

Models can only use a site prior if the site was seen in training. EoS.9 builds its priors and site embeddings from the **fully labelled** files, and maps unseen sites to a shared index 0. This section lists test sites and hours the training set does not cover.

In [ ]:
labels_raw = pd.read_csv(BASE / "train_soundscapes_labels.csv").drop_duplicates()
win = labels_raw.groupby(["filename", "start", "end"]).size().reset_index()
wpf = win.groupby("filename").size()
train_all  = pd.DataFrame({"filename": wpf.index})   # all labelled files
train_all  = pd.concat([train_all, train_all["filename"].apply(parse_fname).apply(pd.Series)], axis=1)
train_full = train_all[train_all["filename"].isin(wpf[wpf == N_WINDOWS].index)]

def summarize(name, df):
    return {"set": name, "files": len(df), "sites": df["site"].nunique(), "hours covered": df["hour_utc"].nunique(),
            "first date": df["date"].min(), "last date": df["date"].max()}

cmp = pd.DataFrame([summarize("test (or dry-run) files", files),
                    summarize("train: all labelled files", train_all),
                    summarize("train: fully labelled files", train_full)])
display(cmp)

test_sites, full_sites, all_sites = set(files["site"]), set(train_full["site"]), set(train_all["site"])
print("Test sites not in the fully labelled training files:", sorted(test_sites - full_sites) or "none")
print("Test sites not in any labelled training file      :", sorted(test_sites - all_sites) or "none")
print("Test hours not in the fully labelled training files:", sorted(set(files["hour_utc"]) - set(train_full["hour_utc"])) or "none")
if DRY_RUN:
    print("\n(DRY RUN: the 'test' files are training files, so overlap here says nothing about the real test set.)")

site_counts = pd.DataFrame({"test": files["site"].value_counts(), "train fully labelled": train_full["site"].value_counts(),
                            "train all labelled": train_all["site"].value_counts()}).fillna(0).astype(int)
ax = site_counts.plot.bar(figsize=(8, 3), color=["#B5533C", "#1F3A5F", "#2A7F8E"])
ax.set_ylabel("Files"); ax.set_title("Files per site"); plt.tight_layout(); plt.show()

## 6. Optional: is the Perch ONNX model available?

EoS.9 turns each 5-second window into a 1536-d embedding and class logits with the Perch v2 ONNX model (`perch_v2_no_dft.onnx`). This cell only looks for the file and prints its input and output signature. It does not run inference.

In [ ]:
onnx_paths = []
root = Path("/kaggle/input")
if root.exists():
    onnx_paths = sorted(root.rglob("perch_v2*.onnx"))[:3]

if not onnx_paths:
    print("No perch_v2*.onnx file found under /kaggle/input, skipping.")
else:
    print("Found:", [str(p) for p in onnx_paths])
    try:
        import onnxruntime as ort
        sess = ort.InferenceSession(str(onnx_paths[0]), providers=["CPUExecutionProvider"])
        print("Inputs :", [(i.name, i.shape, i.type) for i in sess.get_inputs()])
        print("Outputs:", [(o.name, o.shape, o.type) for o in sess.get_outputs()])
    except Exception as e:
        print("onnxruntime could not open the model:", str(e)[:200])

## 7. Submission validator

The functions below reproduce the checks EoS.9 runs before writing `submission.csv`, plus a few more:

* `validate_submission(df, sample, expected_ids)` returns a pass/fail table.
* `compare_row_ids(paths)` compares the `row_id` sets of several per-model files. EoS.9 blends `subm_22.csv`, `subm_51p.csv` and `subm_74.csv` by adding DataFrames, and pandas aligns them by index. If the `row_id` sets differ, the sum contains NaN. This is what the saved dry-run of EoS.9 shows (243 rows and `mean=nan`).

In [ ]:
def validate_submission(df, sample, expected_ids=None, name="submission"):
    checks = []
    def add(check, ok, detail=""):
        checks.append({"check": check, "ok": bool(ok), "detail": detail})

    label_cols = [c for c in sample.columns if c != "row_id"]
    add("has row_id column", "row_id" in df.columns)
    if "row_id" not in df.columns:
        return pd.DataFrame(checks).assign(file=name)

    prob_cols = [c for c in df.columns if c != "row_id"]
    add("column names and order match sample_submission", list(df.columns) == list(sample.columns),
        f"{len(df.columns)} columns vs {len(sample.columns)}")
    add("number of class columns", len(prob_cols) == len(label_cols), f"{len(prob_cols)} vs {len(label_cols)}")
    add("row_id values are unique", df["row_id"].is_unique, f"{int(df['row_id'].duplicated().sum())} duplicates")

    ids, sample_ids = set(df["row_id"].astype(str)), set(sample["row_id"].astype(str))
    add("row_id set equals sample_submission", ids == sample_ids,
        f"missing {len(sample_ids - ids)}, extra {len(ids - sample_ids)}")
    if expected_ids is not None:
        exp = set(expected_ids)
        add("row_id set equals the ids built from the audio files", ids == exp,
            f"missing {len(exp - ids)}, extra {len(ids - exp)}")

    vals = df[prob_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype="float64")
    add("no missing or non-finite values", np.isfinite(vals).all(), f"{int((~np.isfinite(vals)).sum())} bad cells")
    if np.isfinite(vals).any():
        vmin, vmax = np.nanmin(vals), np.nanmax(vals)
        add("probabilities within [0, 1]", vmin >= 0 and vmax <= 1, f"min {vmin:.4f}, max {vmax:.4f}")
    return pd.DataFrame(checks).assign(file=name)

def compare_row_ids(paths):
    sets = {}
    for p in paths:
        p = Path(p)
        if p.exists():
            sets[p.name] = set(pd.read_csv(p, usecols=["row_id"])["row_id"].astype(str))
    if len(sets) < 2:
        print(f"Need at least two existing files to compare; found {len(sets)}: {list(sets)}")
        return None
    union = set().union(*sets.values()); inter = set.intersection(*sets.values())
    tbl = pd.DataFrame({"rows": {k: len(v) for k, v in sets.items()},
                        "not in all files": {k: len(union - v) for k, v in sets.items()}})
    print(f"Union: {len(union):,} rows  |  in every file: {len(inter):,} rows")
    print("Aligned: " + ("YES, a weighted sum keeps every row" if len(union) == len(inter) else "NO, a weighted sum would create NaN rows"))
    return tbl

print("validate_submission and compare_row_ids defined.")

In [ ]:
# 1) Sanity check: the validator should pass on the sample submission itself.
display(validate_submission(sample, sample, name="sample_submission.csv")[["check", "ok", "detail"]])

# 2) Check any submission files that exist next to the notebook or in /kaggle/working.
search_dirs = [Path("."), Path("/kaggle/working"), OUT_DIR]
found = []
for d in search_dirs:
    for fname in ["submission.csv", "subm_22.csv", "subm_51p.csv", "subm_74.csv"]:
        p = d / fname
        if p.exists() and p.resolve() not in [q.resolve() for q in found]:
            found.append(p)
if not found:
    print("No submission.csv or per-model files found to validate. Run a model or blend first, then re-run this cell.")
for p in found:
    print(f"\n== {p} ==")
    display(validate_submission(pd.read_csv(p), sample, expected_ids=None if DRY_RUN else expected_ids, name=p.name)[["check", "ok", "detail"]])

# 3) Are the per-model files row-aligned?
display(compare_row_ids([p for p in found if p.name.startswith("subm_")]))

## 8. Outputs that depend on the composition of the test set

The final xSED blend in EoS.9 converts each class column to **percentile ranks across all test rows** (`rank(axis=0, pct=True)`) before mixing ProtoSSM and SED. A row's value therefore depends on which other rows are in the test set, unlike a plain probability. The toy example below shows the effect.

In [ ]:
scores = pd.Series([0.10, 0.20, 0.30, 0.40, 0.90], index=[f"win{i}" for i in range(1, 6)], name="raw probability")
full_rank   = scores.rank(pct=True).rename("rank in all 5 windows")
subset_rank = scores.iloc[:3].rank(pct=True).rename("rank in the first 3 windows")
display(pd.concat([scores, full_rank, subset_rank], axis=1))
print("win3 has the same raw probability in both runs, but its rank changes from "
      f"{full_rank['win3']:.2f} to {subset_rank['win3']:.2f}.")
print("Consequence: submit predictions for the complete test set in one run, and do not compare rank-blended "
      "values between runs that used different sets of files.")

## 9. Summary and export

In [ ]:
n_flagged = int(audio[flag_cols].any(axis=1).sum())
summary = {
    "mode": "dry_run" if DRY_RUN else "real_test",
    "files": int(len(files)),
    "expected_rows": int(len(expected_ids)),
    "sample_submission_rows": int(len(sample)),
    "sites": sorted(files["site"].unique().tolist()),
    "hours_covered": int(files["hour_utc"].nunique()),
    "date_range": [str(files["date"].min().date()), str(files["date"].max().date())],
    "files_with_audio_flags": n_flagged,
    "test_sites_not_in_fully_labelled_train": sorted(test_sites - full_sites),
}
display(pd.Series(summary, name="value").to_frame())

with open(OUT_DIR / "test_data_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
files.drop(columns=["date"]).merge(audio.drop(columns=["error"]), on="filename").to_csv(OUT_DIR / "test_files.csv", index=False)
pd.Series(expected_ids, name="row_id").to_csv(OUT_DIR / "expected_row_ids.csv", index=False)
print("Wrote:", sorted(p.name for p in OUT_DIR.glob("*") if p.name in {"test_data_summary.json", "test_files.csv", "expected_row_ids.csv"}))

## Notes

* **Dry run is not evidence of quality.** In the saved EoS.9 run, the 3-row placeholder submission is filled with the per-class mean of the dry-run predictions, so all rows are identical. It only shows that the code runs and the format is right.
* **Unseen sites.** A test site missing from the fully labelled training files gets no site-specific prior and shares one embedding index with every other unseen site.
* **Row alignment.** The EoS.9 per-model cells assert that their `row_id` set equals `sample_submission.csv`, but the final weighted sum of the three model files does not re-check alignment. Use `compare_row_ids` before blending.